# So sánh Quantum K-Means theo số chiều (Dims)
**So sánh inertia, silhouette, davies-bouldin, calinski-harabasz, thời gian hội tụ, và heatmap k×dims**

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in [
    "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn",
    "qiskit==1.4.2",
    "qiskit-aer==0.15.1",
    "qiskit-machine-learning==0.7.2"
]:
    try:
        install(pkg)
        print(f"{pkg} installed success !")
    except Exception as e:
        print(f"{pkg} — {e}")

print("\nFinish.")

In [ ]:
# ==============================================================================
# CELL 1: Import + Setup
# ==============================================================================
import warnings, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
warnings.filterwarnings('ignore')

COLOR_BASE = '#1a1a2e'
BG = '#f8f9fa'
PALETTE = ['#e94560','#0f3460','#533483','#e8c547',
           '#16213e','#a8dadc','#457b9d','#f4a261','#2a9d8f']

print('Setup OK')

In [ ]:
# ==============================================================================
# CELL 1.5: Định nghĩa class để pickle load được
# ==============================================================================
from sklearn.cluster import KMeans as _SKLearnKMeans

class ClassicalKMeansWithHistory:
    def __init__(self, n_clusters, n_init=30, max_iter=300, random_state=123):
        self.n_clusters, self.n_init = n_clusters, n_init
        self.max_iter, self.random_state = max_iter, random_state
        self.labels_ = self.cluster_centers_ = self.inertia_ = None
        self.best_run_history_ = []
        self.convergence_history_ = []
    def fit(self, X):
        pass

class QiskitSwapTestKMeans:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
        self.labels_ = None
        self.cluster_centers_ = None
        self.inertia_ = None
        self.best_run_history_ = []
        self.convergence_history_ = []

print('Classes defined OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_PATH  = '/content/drive/MyDrive/...'   # path đến artifacts
CHECKPOINT_DIR = '/content/drive/MyDrive/...'   # path đến checkpoints

with open(ARTIFACT_PATH, 'rb') as f:
    artifacts = pickle.load(f)

pca_scores = artifacts['pca_scores']
pca_cutoff = artifacts['pca_cutoff']
X_full     = pca_scores.values.astype(float)

K_RANGE    = list(range(2, 11))
DIMS_LIST  = [2, 5, 10, 15, 20, pca_cutoff]

print(f'X_full shape : {X_full.shape}')
print(f'pca_cutoff   : {pca_cutoff}')
print(f'DIMS_LIST    : {DIMS_LIST}')

In [ ]:
# ==============================================================================
# CELL 3: Load tất cả checkpoint dims
# ==============================================================================
def load_ckpt(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

results = {}

for dims in DIMS_LIST:
    results[dims] = {}
    for k in K_RANGE:
        # Luôn dùng _d{dims}, fallback sang baseline nếu không có
        ckpt = load_ckpt(f'quantum_k{k}_d{dims}')
        if ckpt is None:
            ckpt = load_ckpt(f'quantum_k{k}')  # fallback baseline

        if ckpt is not None:
            results[dims][k] = ckpt

# Tổng hợp
print('Trang thai checkpoint:')
for dims in DIMS_LIST:
    found = sorted(results[dims].keys())
    missing = [k for k in K_RANGE if k not in found]
    print(f'  dims={dims:3d} | found k={found} | missing k={missing}')

In [ ]:
# ==============================================================================
# CELL 4: Tính metrics cho từng (dims, k)
# ==============================================================================
records = []

for dims in DIMS_LIST:
    X = X_full[:, :dims]
    from sklearn.preprocessing import normalize
    X_norm = normalize(X.astype(float), norm='l2')

    for k in K_RANGE:
        if k not in results[dims]:
            continue
        m = results[dims][k]
        labels = m.labels_

        # Một số trường hợp có thể chỉ có 1 cluster sau khi hội tụ kém -> skip metrics cần >=2 cluster
        n_unique = len(np.unique(labels))
        sil = silhouette_score(X_norm, labels) if n_unique > 1 else np.nan
        dbi = davies_bouldin_score(X_norm, labels) if n_unique > 1 else np.nan
        chi = calinski_harabasz_score(X_norm, labels) if n_unique > 1 else np.nan

        records.append(dict(
            dims              = dims,
            k                 = k,
            inertia           = m.inertia_,
            silhouette        = sil,
            davies_bouldin    = dbi,
            calinski_harabasz = chi,
            n_iters           = len(m.best_run_history_),
            n_unique_clusters = n_unique,
        ))

dims_df = pd.DataFrame(records)
print(dims_df.round(4).to_string(index=False))

In [ ]:
# ==============================================================================
# CELL 5: Fig 4 — Silhouette & Davies-Bouldin theo dims (mỗi k 1 đường)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG)

for ax, col, title, ylabel, better in [
    (axes[0], 'silhouette',     'Silhouette Score theo Dims',     'Score', 'high'),
    (axes[1], 'davies_bouldin', 'Davies-Bouldin theo Dims',       'Index', 'low'),
]:
    ax.set_facecolor('#fff')
    for idx, k in enumerate(K_RANGE):
        sub = dims_df[dims_df.k == k].sort_values('dims')
        if sub.empty:
            continue
        ax.plot(sub['dims'], sub[col], 'o-', color=PALETTE[idx % len(PALETTE)],
                lw=1.5, ms=5, label=f'k={k}', alpha=0.85)
    note = 'cao hơn tốt hơn' if better == 'high' else 'thấp hơn tốt hơn'
    ax.set_title(f'{title}\n{note}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Số chiều (dims)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(DIMS_LIST)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', ls='--', alpha=0.4, color='#ccc')
    ax.legend(fontsize=8, ncol=3)

fig.suptitle('Ảnh hưởng của số chiều đến chất lượng Clustering (Quantum)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_dims_quality.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig4 done')

In [ ]:
# ==============================================================================
# CELL 6: Fig 5 — Inertia & Calinski-Harabasz theo dims
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG)

for ax, col, title, ylabel, better in [
    (axes[0], 'inertia',           'Inertia theo Dims',           'Inertia', 'low'),
    (axes[1], 'calinski_harabasz', 'Calinski-Harabasz theo Dims', 'Index',   'high'),
]:
    ax.set_facecolor('#fff')
    for idx, k in enumerate(K_RANGE):
        sub = dims_df[dims_df.k == k].sort_values('dims')
        if sub.empty:
            continue
        ax.plot(sub['dims'], sub[col], 's--', color=PALETTE[idx % len(PALETTE)],
                lw=1.5, ms=5, label=f'k={k}', alpha=0.85)
    note = '↓ thấp hơn tốt hơn' if better == 'low' else '↑ cao hơn tốt hơn'
    ax.set_title(f'{title}\n{note}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Số chiều (dims)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(DIMS_LIST)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', ls='--', alpha=0.4, color='#ccc')
    ax.legend(fontsize=8, ncol=3)

fig.suptitle('Inertia & Calinski-Harabasz theo số chiều (Quantum)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_dims_inertia_chi.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig5 done')

In [ ]:
# ==============================================================================
# CELL 7: Fig 6 — Heatmap k x dims cho từng metric
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor(BG)

metric_specs = [
    ('silhouette',        'Silhouette (cao = tốt)',       'YlGn'),
    ('davies_bouldin',    'Davies-Bouldin (thấp = tốt)',  'YlOrRd_r'),
    ('inertia',           'Inertia (thấp = tốt)',         'YlOrRd_r'),
    ('calinski_harabasz', 'Calinski-Harabasz (cao = tốt)','YlGn'),
]

for ax, (col, title, cmap) in zip(axes.flat, metric_specs):
    pivot = dims_df.pivot(index='k', columns='dims', values=col)
    pivot = pivot.reindex(index=K_RANGE, columns=DIMS_LIST)

    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(DIMS_LIST)))
    ax.set_xticklabels(DIMS_LIST)
    ax.set_yticks(range(len(K_RANGE)))
    ax.set_yticklabels(K_RANGE)
    ax.set_xlabel('Dims', fontsize=10)
    ax.set_ylabel('k', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=8, color='black')
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Heatmap k × Dims cho từng Metric (Quantum)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig6_heatmap_k_dims.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig6 done')

In [ ]:
# ==============================================================================
# CELL 8: Fig 7 — Số iterations để hội tụ theo dims (proxy cho thời gian)
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('#fff')

for idx, k in enumerate(K_RANGE):
    sub = dims_df[dims_df.k == k].sort_values('dims')
    if sub.empty:
        continue
    ax.plot(sub['dims'], sub['n_iters'], 'o-', color=PALETTE[idx % len(PALETTE)],
            lw=1.5, ms=5, label=f'k={k}', alpha=0.85)

ax.set_title('Số iterations để hội tụ theo Dims\n'
              '(proxy cho thời gian tính toán: ít iter = nhanh hơn)',
              fontsize=12, fontweight='bold')
ax.set_xlabel('Số chiều (dims)', fontsize=10)
ax.set_ylabel('Số iterations (best run)', fontsize=10)
ax.set_xticks(DIMS_LIST)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', ls='--', alpha=0.4, color='#ccc')
ax.legend(fontsize=8, ncol=3)

plt.tight_layout()
plt.savefig('fig7_dims_convergence_iters.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig7 done')

In [ ]:
# ==============================================================================
# CELL 9: Kết luận tự động
# ==============================================================================
print('='*60)
print('  KET QUA SO SANH THEO DIMS')
print('='*60)

# Dims tốt nhất theo silhouette trung bình
avg_by_dims = dims_df.groupby('dims')[['silhouette','davies_bouldin',
                                          'calinski_harabasz','inertia',
                                          'n_iters']].mean()
print('\nTrung binh cac metric theo dims:')
print(avg_by_dims.round(4).to_string())

best_dims_sil = avg_by_dims['silhouette'].idxmax()
best_dims_dbi = avg_by_dims['davies_bouldin'].idxmin()
fastest_dims  = avg_by_dims['n_iters'].idxmin()

print(f'\nDims tot nhat theo Silhouette       : {best_dims_sil}')
print(f'Dims tot nhat theo Davies-Bouldin   : {best_dims_dbi}')
print(f'Dims hoi tu nhanh nhat (it iter)    : {fastest_dims}')

print('\nNhan xet:')
if best_dims_sil == DIMS_LIST[-1]:
    print('  -> So chieu cao (pca_cutoff) cho ket qua clustering tot nhat')
else:
    print(f'  -> So chieu {best_dims_sil} cho ket qua tot hon pca_cutoff'
          f' ({DIMS_LIST[-1]}) — co the do curse of dimensionality')

if fastest_dims == DIMS_LIST[0]:
    print('  -> So chieu thap nhat hoi tu nhanh nhat (it qubit hon)')

In [ ]:
# ==============================================================================
# CELL 10: Lưu kết quả
# ==============================================================================
dims_df.to_csv('dims_comparison_metrics.csv', index=False)

with open('dims_comparison_results.pkl', 'wb') as f:
    pickle.dump(dict(
        dims_df   = dims_df,
        results   = results,
        DIMS_LIST = DIMS_LIST,
        K_RANGE   = K_RANGE,
    ), f)

print('Da luu:')
for fn in [
    'dims_comparison_metrics.csv',
    'dims_comparison_results.pkl',
    'fig4_dims_quality.png',
    'fig5_dims_inertia_chi.png',
    'fig6_heatmap_k_dims.png',
    'fig7_dims_convergence_iters.png',
]:
    print(f'  {fn}')

---
## So sánh Classical vs Quantum K-Means theo Dims
**Load classical checkpoints tại từng dims, tính metrics, vẽ ARI + gap metric**

In [ ]:
# ==============================================================================
# CELL A: Load Classical checkpoints theo dims
# Giả sử checkpoint đặt tên: classical_k{k}_d{dims}.pkl
# Fallback về classical_k{k}.pkl (baseline dims = pca_cutoff)
# ==============================================================================
classical_results_by_dims = {}

for dims in DIMS_LIST:
    classical_results_by_dims[dims] = {}
    for k in K_RANGE:
        ckpt = load_ckpt(f'classical_k{k}_d{dims}')
        if ckpt is None:
            ckpt = load_ckpt(f'classical_k{k}')   # fallback
        if ckpt is not None:
            classical_results_by_dims[dims][k] = ckpt

print('Classical checkpoints theo dims:')
for dims in DIMS_LIST:
    found   = sorted(classical_results_by_dims[dims].keys())
    missing = [k for k in K_RANGE if k not in classical_results_by_dims[dims]]
    print(f'  dims={dims:3d} | found k={found} | missing k={missing}')


In [ ]:
# ==============================================================================
# CELL B: Tính metrics Classical + ARI(Classical, Quantum) theo từng (dims, k)
# ==============================================================================
from sklearn.preprocessing import normalize

COLOR_CL  = '#1a1a2e'
COLOR_QK  = '#e94560'
COLOR_ARI = '#0f3460'

compare_records = []

for dims in DIMS_LIST:
    X      = X_full[:, :dims]
    X_norm = normalize(X.astype(float), norm='l2')

    for k in K_RANGE:
        qkm = results[dims].get(k)
        cl  = classical_results_by_dims[dims].get(k)
        if qkm is None and cl is None:
            continue

        def safe_metrics(m):
            if m is None:
                return dict(inertia=float('nan'), silhouette=float('nan'),
                            davies_bouldin=float('nan'), calinski_harabasz=float('nan'))
            lbl = m.labels_
            nu  = len(set(lbl))
            return dict(
                inertia           = m.inertia_,
                silhouette        = silhouette_score(X_norm, lbl)        if nu > 1 else float('nan'),
                davies_bouldin    = davies_bouldin_score(X_norm, lbl)    if nu > 1 else float('nan'),
                calinski_harabasz = calinski_harabasz_score(X_norm, lbl) if nu > 1 else float('nan'),
            )

        qm = safe_metrics(qkm)
        cm = safe_metrics(cl)

        ari = (adjusted_rand_score(cl.labels_, qkm.labels_)
               if cl is not None and qkm is not None else float('nan'))

        compare_records.append(dict(
            dims=dims, k=k, ari=ari,
            # gap = QKM - Classical  (dương = QKM tốt hơn khi metric là 'high')
            gap_silhouette        = qm['silhouette']        - cm['silhouette'],
            gap_davies_bouldin    = cm['davies_bouldin']    - qm['davies_bouldin'],  # đổi dấu: thấp tốt
            gap_calinski_harabasz = qm['calinski_harabasz'] - cm['calinski_harabasz'],
            gap_inertia           = cm['inertia']           - qm['inertia'],          # đổi dấu
            **{f'cl_{kk}': vv  for kk, vv in cm.items()},
            **{f'qkm_{kk}': vv for kk, vv in qm.items()},
        ))

cmp_df = pd.DataFrame(compare_records)
print(cmp_df[['dims','k','ari',
              'cl_silhouette','qkm_silhouette',
              'gap_silhouette']].round(4).to_string(index=False))


In [ ]:
# ==============================================================================
# CELL C: Fig 8 — ARI(dims) mỗi k một đường
# ARI = 1 => QKM và Classical ra nhãn giống nhau hoàn toàn
# ==============================================================================
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('#fff')

for idx, k in enumerate(K_RANGE):
    sub = cmp_df[cmp_df.k == k].sort_values('dims')
    if sub.empty or sub['ari'].isna().all():
        continue
    ax.plot(sub['dims'], sub['ari'], 'o-',
            color=PALETTE[idx % len(PALETTE)], lw=1.8, ms=6,
            label=f'k={k}', alpha=0.88)

ax.axhline(1.0, color='gray', lw=1, ls=':', alpha=0.5, label='ARI=1 (đồng nhất)')
ax.axhline(0.0, color='red',  lw=1, ls=':', alpha=0.3, label='ARI=0 (ngẫu nhiên)')
ax.set_title('ARI giữa Classical và QKM theo số chiều\n'
             '(càng gần 1 = hai thuật toán ra kết quả càng giống nhau)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Số chiều (dims)', fontsize=10)
ax.set_ylabel('Adjusted Rand Index', fontsize=10)
ax.set_ylim(-0.05, 1.15)
ax.set_xticks(DIMS_LIST)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', ls='--', alpha=0.4, color='#ccc')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.savefig('fig8_ari_by_dims.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig8 done')


In [ ]:
# ==============================================================================
# CELL D: Fig 9 — Gap metric theo dims  (QKM - Classical, chuẩn hoá về cùng hướng)
# gap > 0  =>  QKM TỐT HƠN classical tại dims đó
# gap < 0  =>  Classical tốt hơn
# ==============================================================================
gap_cols = [
    ('gap_silhouette',        'Silhouette gap\n(QKM − CL)',        'high'),
    ('gap_davies_bouldin',    'Davies-Bouldin gap\n(CL − QKM)',    'high'),
    ('gap_calinski_harabasz', 'Calinski-Harabasz gap\n(QKM − CL)','high'),
    ('gap_inertia',           'Inertia gap\n(CL − QKM)',           'high'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.patch.set_facecolor(BG)

for ax, (col, title, _) in zip(axes.flat, gap_cols):
    ax.set_facecolor('#fff')
    for idx, k in enumerate(K_RANGE):
        sub = cmp_df[cmp_df.k == k].sort_values('dims')
        if sub.empty:
            continue
        ax.plot(sub['dims'], sub[col], 'o-',
                color=PALETTE[idx % len(PALETTE)], lw=1.5, ms=5,
                label=f'k={k}', alpha=0.85)
    ax.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6)
    ax.fill_between(DIMS_LIST, 0, 0, alpha=0)   # placeholder
    ax.set_title(f'{title}\n(> 0 => QKM tốt hơn)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Dims', fontsize=10)
    ax.set_ylabel('Gap', fontsize=10)
    ax.set_xticks(DIMS_LIST)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', ls='--', alpha=0.35, color='#ccc')
    ax.legend(fontsize=8, ncol=3)

fig.suptitle('Gap metric (QKM − Classical) theo số chiều\n'
             'Vùng > 0: QKM chiếm ưu thế | Vùng < 0: Classical chiếm ưu thế',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig9_gap_by_dims.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig9 done')


In [ ]:
# ==============================================================================
# CELL E: Fig 10 — Side-by-side line chart Classical vs QKM cho 4 metrics
# mỗi subplot: 1 metric, 2 đường (CL + QKM), trục x = dims, hue = k
# ==============================================================================
import matplotlib.lines as mlines

metric_pairs = [
    ('cl_silhouette',        'qkm_silhouette',        'Silhouette Score',        'cao hơn tốt hơn'),
    ('cl_davies_bouldin',    'qkm_davies_bouldin',    'Davies-Bouldin Index',    'thấp hơn tốt hơn'),
    ('cl_calinski_harabasz', 'qkm_calinski_harabasz', 'Calinski-Harabasz Index', 'cao hơn tốt hơn'),
    ('cl_inertia',           'qkm_inertia',           'Inertia',                 'thấp hơn tốt hơn'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(BG)

for ax, (cl_col, qkm_col, title, note) in zip(axes.flat, metric_pairs):
    ax.set_facecolor('#fff')
    for idx, k in enumerate(K_RANGE):
        sub = cmp_df[cmp_df.k == k].sort_values('dims')
        if sub.empty:
            continue
        c = PALETTE[idx % len(PALETTE)]
        ax.plot(sub['dims'], sub[cl_col],  'o-',  color=c, lw=1.5, ms=5, alpha=0.9)
        ax.plot(sub['dims'], sub[qkm_col], 's--', color=c, lw=1.5, ms=5, alpha=0.55)
    ax.set_title(f'{title}\n{note}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Dims', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_xticks(DIMS_LIST)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', ls='--', alpha=0.35, color='#ccc')

# Legend chung: màu = k, style = method
k_handles  = [mlines.Line2D([],[], color=PALETTE[i % len(PALETTE)], lw=2, label=f'k={k}')
               for i, k in enumerate(K_RANGE)]
m_handles  = [
    mlines.Line2D([],[], color='gray', lw=2, ls='-',  marker='o', label='Classical'),
    mlines.Line2D([],[], color='gray', lw=2, ls='--', marker='s', label='Quantum',  alpha=0.6),
]
fig.legend(handles=k_handles + m_handles,
           loc='lower center', ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, -0.03), frameon=True)

fig.suptitle('Classical vs QKM theo số chiều\n',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig10_side_by_side_dims.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig10 done')


In [ ]:
# ==============================================================================
# CELL F: Fig 11 — Heatmap ARI k × dims
# ==============================================================================
pivot_ari = cmp_df.pivot(index='k', columns='dims', values='ari')
pivot_ari = pivot_ari.reindex(index=K_RANGE, columns=DIMS_LIST)

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG)

im = ax.imshow(pivot_ari.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(DIMS_LIST))); ax.set_xticklabels(DIMS_LIST)
ax.set_yticks(range(len(K_RANGE)));   ax.set_yticklabels(K_RANGE)
ax.set_xlabel('Dims', fontsize=11)
ax.set_ylabel('k',    fontsize=11)
ax.set_title('Heatmap ARI (Classical vs QKM) theo k × Dims\n'
             'Xanh = đồng thuận cao | Đỏ = nhãn khác nhau nhiều',
             fontsize=12, fontweight='bold')

for i in range(pivot_ari.shape[0]):
    for j in range(pivot_ari.shape[1]):
        val = pivot_ari.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9,
                    color='black' if 0.3 < val < 0.8 else 'white')

plt.colorbar(im, ax=ax, shrink=0.85, label='ARI')
plt.tight_layout()
plt.savefig('fig11_heatmap_ari_k_dims.png', dpi=150, bbox_inches='tight')
plt.show(); print('fig11 done')


In [ ]:
# ==============================================================================
# CELL G: Kết luận so sánh Classical vs QKM theo dims
# ==============================================================================
print('='*65)
print('  KET QUA SO SANH CLASSICAL vs QKM THEO DIMS')
print('='*65)

avg = cmp_df.groupby('dims')[['ari','gap_silhouette','gap_davies_bouldin',
                               'gap_calinski_harabasz','gap_inertia']].mean()
print('\nTrung binh theo dims (trung binh tren moi k):')
print(avg.round(4).to_string())

best_ari_dims  = avg['ari'].idxmax()
qkm_win_dims   = avg[avg['gap_silhouette'] > 0].index.tolist()
cl_win_dims    = avg[avg['gap_silhouette'] <= 0].index.tolist()

print(f'\nDims ma QKM va Classical dong nhat (ARI cao nhat) : {best_ari_dims}')
print(f'Dims QKM tot hon Classical (Silhouette gap > 0)   : {qkm_win_dims}')
print(f'Dims Classical tot hon QKM (Silhouette gap <= 0)  : {cl_win_dims}')

# Lưu thêm
cmp_df.to_csv('classical_vs_qkm_by_dims.csv', index=False)
print('\nDa luu: classical_vs_qkm_by_dims.csv')
